In [1]:
import re
import glob
import zipfile
from pathlib import Path
import json
import requests
import pandas as pd
from urllib.request import urlretrieve

In [2]:
import time, psutil, os, functools

_proc = psutil.Process(os.getpid())

def measure(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        m0 = _proc.memory_info().rss / 1e6
        c0 = time.process_time()
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        print(f"{fn.__name__}:  wall={time.perf_counter()-t0:.1f}s  "
              f"cpu={time.process_time()-c0:.1f}s  "
              f"mem Δ{_proc.memory_info().rss/1e6 - m0:+.0f} MB")
        return out
    return wrapper

In [3]:
WORKDIR = Path.cwd()
OUTPUT_DIR = WORKDIR / "figsharerainfall"
OUTPUT_DIR.mkdir(exist_ok=True)
FORCE_DOWNLOAD = False 

In [4]:
# Necessary metadata
article_id = 14096681
url = f"https://api.figshare.com/v2/articles/{article_id}"
headers = {"Content-Type": "application/json"}
output_directory = "figsharerainfall/"

In [5]:
response = requests.request("GET", url, headers=headers)
data = json.loads(response.text)
files = data["files"]
files

[{'id': 26579150,
  'name': 'daily_rainfall_2014.png',
  'size': 58863,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579150',
  'supplied_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'computed_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'mimetype': 'image/png'},
 {'id': 26579171,
  'name': 'environment.yml',
  'size': 192,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579171',
  'supplied_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'computed_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'mimetype': 'text/plain'},
 {'id': 26586554,
  'name': 'README.md',
  'size': 5422,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26586554',
  'supplied_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'computed_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'mimetype': 'text/x-python'},
 {'id': 26766812,
  'name': 'data.zip',
  'size': 814041183,
  'is_link_only': False,
  'download_url': 'https://n

In [6]:
files_to_dl = ["data.zip"]
for file in files:
    if file["name"] in files_to_dl:

        output_path = OUTPUT_DIR / file["name"]

        if output_path.exists() and not FORCE_DOWNLOAD:
            print(f"{file['name']} already exists. Skipping.")
        else:
            output_path.parent.mkdir(exist_ok=True)

            print(f"Downloading {file['name']}...")
            urlretrieve(file["download_url"], output_path)

In [12]:
zip_path = OUTPUT_DIR / "data.zip"
extract_dir = OUTPUT_DIR / "data"

if not extract_dir.exists():
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extractall(extract_dir)
else:
    print("Files already extracted. Skipping extraction.")

Extracting files...


In [14]:
%ls -ltr figsharerainfall/data/

total 10483376
-rw-r--r--@  1 oswingan  staff   95376895 Mar 25 16:58 MPI-ESM-1-2-HAM_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff   94960113 Mar 25 16:58 AWI-ESM-1-1-LR_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff   82474546 Mar 25 16:58 NorESM2-LM_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff  127613760 Mar 25 16:58 ACCESS-CM2_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff  232118894 Mar 25 16:58 FGOALS-f3-L_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff  330360682 Mar 25 16:58 CMCC-CM2-HR4_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff  254009247 Mar 25 16:58 MRI-ESM2-0_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff  235661418 Mar 25 16:58 GFDL-CM4_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff  294260911 Mar 25 16:58 BCC-CSM2-MR_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff  295768615 Mar 25 16:58 EC-Earth3-Veg-LR_daily_rainfall_NSW.csv
-rw-r--r--@  1 oswingan  staff  328852379 Mar 25 16:58 CMCC-ESM2_daily_rainfal

In [24]:
@measure
def combine_csvs(folder):
    folder = Path(folder)
    csv_files = [
        file for file in glob.glob(str(folder / "*.csv"))
        if Path(file).name != "observed_daily_rainfall_SYD.csv"
    ]
    
    dfs = []
    for filepath in csv_files:
        filename = Path(filepath).name
        model = re.search(r"([^_]*)", filename).group(1)
        df = pd.read_csv(filepath)
        df["model"] = model
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    return combined

In [27]:
combined_df = combine_csvs(OUTPUT_DIR / "data")
combined_df.to_csv(OUTPUT_DIR / "combined_rainfall.csv", index=False)
combined_df

combine_csvs:  wall=32.4s  cpu=30.7s  mem Δ+2398 MB


,time,lat_min,lat_max,lon_min,lon_max,rain (mm/day),model
0,1889-01-01 12:00:00,-35.439867,-33.574619,141.5625,143.4375,4.244226e-13,MPI-ESM-1-2-HAM
1,1889-01-02 12:00:00,-35.439867,-33.574619,141.5625,143.4375,4.217326e-13,MPI-ESM-1-2-HAM
2,1889-01-03 12:00:00,-35.439867,-33.574619,141.5625,143.4375,4.498125e-13,MPI-ESM-1-2-HAM
3,1889-01-04 12:00:00,-35.439867,-33.574619,141.5625,143.4375,4.251282e-13,MPI-ESM-1-2-HAM
4,1889-01-05 12:00:00,-35.439867,-33.574619,141.5625,143.4375,4.270161e-13,MPI-ESM-1-2-HAM
...,...,...,...,...,...,...,...
62467838,2014-12-27 12:00:00,-30.157068,-29.214660,153.1250,154.3750,6.689683e+00,SAM0-UNICON
62467839,2014-12-28 12:00:00,-30.157068,-29.214660,153.1250,154.3750,7.862555e+00,SAM0-UNICON
62467840,2014-12-29 12:00:00,-30.157068,-29.214660,153.1250,154.3750,1.000503e+01,SAM0-UNICON
62467841,2014-12-30 12:00:00,-30.157068,-29.214660,153.1250,154.3750,8.541592e+00,SAM0-UNICON


In [ ]:
# Remove uncombined data to free up hard drive space
import shutil
shutil.rmtree(OUTPUT_DIR / "data")